# Tools

Models can request to call tools that perform tasks such as fetching data from a a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and argument definitions(often as JSON schema).
2. A function or coroutine to execute.

In [1]:
# Initializing the model
import os

from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")
response = model.invoke("Why do LLM hallucinate?")
response

AIMessage(content='### Short answer  \nLarge language models (LLMs) “hallucinate” because they are trained to **predict the next token in a sequence** based on statistical patterns in their training data, not to verify facts or to consult an external knowledge base. When the model is asked to generate information that it hasn’t seen or that conflicts with the data distribution it has learned, it will still produce something that *looks* plausible in the language it has been trained on. That output is often fluent, but it can be wrong, fabricated, or out‑of‑context.\n\n---\n\n## 1. The root cause: a next‑token probability model\n\n| What the model does | What we *want* it to do |\n|---------------------|--------------------------|\n| Predict the most likely next word given the preceding context. | Produce factually accurate, grounded statements. |\n\n- **Training objective**: minimize cross‑entropy loss over millions of token sequences.  \n- **Resulting behavior**: The model learns *pat

## Syntax of a Tool

> LLM identifies the functionality of a tool via its **docstring**.

```python
@tool
def my_tool(input: str) -> str:
    """Describe what this tool does — the LLM reads this."""
    return f"Processed: {input}"
```

In [14]:
from langchain.tools import tool

@tool
def get_weather(location:str)-> str:
    """Get the current weather for a given location."""
    return f"The weather in {location} is sunny."

model_with_tools = model.bind_tools([get_weather])


In [24]:
# Calling the tool

response = model_with_tools.invoke("Get the weather for New York")
print(response)

for tool_call in response.tool_calls:
    # View tool calls made by model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'We need to call get_weather function.', 'tool_calls': [{'id': 'fc_94c863a7-ba46-421f-bb87-67f054473bae', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 128, 'total_tokens': 161, 'completion_time': 0.035453544, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.007330924, 'prompt_tokens_details': None, 'queue_time': 0.39602959, 'total_time': 0.042784468}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e11c8bbf69', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a16e33-b262-72d0-9e08-73fa02ff4d79-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'fc_94c863a7-ba46-421f-bb87-67f054473bae', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 128, 'output_tok

### Tool Execution Loops

In [25]:
# Step 1 : Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2 : Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #  Execute the tool with generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3 : Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston is sunny.


In [26]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call get_weather function.', 'tool_calls': [{'id': 'fc_7a558458-42a2-4757-8b0a-5499c9f642d0', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 128, 'total_tokens': 160, 'completion_time': 0.034405711, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.006171025, 'prompt_tokens_details': None, 'queue_time': 0.34602264, 'total_time': 0.040576736}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a4315eb300', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a16e33-bd2e-73f3-8741-e4a1b65bac42-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_7a558458-42a2-4757-8b0a-5499c9f642d0', 'type': 'tool_call